# Gym Churn: Customer Clustering

This notebook groups Model Fitness customers into clusters based on their characteristics, using hierarchical clustering to estimate the number of groups and K-means to assign customers to them. Each cluster is then profiled and compared by churn rate to identify which customer groups are most likely to leave and which are most loyal.

In [1]:
# Libraries for data handling, clustering and visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)  # show all columns in outputs

## 1. Data Preparation for Clustering

In [2]:
# Load the cleaned dataset prepared in notebook 01
gym_data = pd.read_csv('../data/gym_churn_clean.csv')
gym_data.head()

,gender,near_location,partner,promo_friends,phone,contract_period,group_visits,age,avg_additional_charges_total,month_to_end_contract,lifetime,avg_class_frequency_total,avg_class_frequency_current_month,churn
0,1,1,1,1,0,6,1,29,14.227470,5,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1,3,1.113884,1.120078,0


In [3]:
# Set the churn column aside: clusters are built only from customer characteristics
# One feature from each highly correlated pair is dropped, as in notebook 02
X = gym_data.drop(
    columns=['churn', 'month_to_end_contract', 'avg_class_frequency_total'])

print(X.shape)

(4000, 11)


In [4]:
# Standardize features so that all of them contribute equally to the distances
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Observations
- The churn column is set aside, so clusters are based only on customer characteristics; churn is used later to compare the clusters.
- As in the modeling notebook, `month_to_end_contract` and `avg_class_frequency_total` are excluded, since they almost duplicate other features and would be counted twice in the distance calculations.
- All features are standardized so that each contributes equally to the distances between customers. The scaler is fitted on the full dataset, since clustering does not involve a validation set.